# Procesamiento de Imágenes con OpenCV

## Filtrado y Convolución

El filtrado de imágenes aplica un **kernel** (pequeña matriz) a cada píxel y sus vecinos.
Un filtro de imagen clásico y una capa convolucional ambos deslizan una pequeña matriz por la imagen y combinan los píxeles cercanos. La diferencia está en el origen del kernel.

- En el **filtrado con OpenCV**, elegimos el kernel nosotros mismos porque queremos un efecto específico como desenfoque, enfoque o detección de bordes.
- En una **CNN**, la red aprende automáticamente los valores del kernel a partir de datos etiquetados.

La operación es por tanto estrechamente relacionada, pero la configuración de aprendizaje es diferente: los filtros fijos son herramientas de diseño manual, mientras que los filtros de CNN son características entrenables.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

sample_path = "../resources/images/lenna.png"
img = cv2.imread(sample_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

### Filtros de Desenfoque

El desenfoque reduce el ruido y los detalles finos reemplazando cada píxel con una combinación ponderada de sus vecinos. El **tamaño del kernel** controla la intensidad — siempre un número impar (3, 5, 7...) para que haya un píxel central bien definido.

| Filtro | Idea clave | Mejor uso |
|--------|----------|---------| 
| **Promedio** | Media simple de todos los vecinos | Rápido, de propósito general |
| **Gaussiano** | Media ponderada — el píxel central cuenta más | La mejor reducción de ruido de propósito general |
| **Mediana** | Valor mediano de los vecinos (no media) | Elimina ruido sal y pimienta sin desenfocar los bordes |
| **Bilateral** | Gaussiano, pero ignora píxeles con brillo muy diferente | Eliminación de ruido **preservando los bordes** |

In [ ]:
# El tamaño del kernel debe ser un entero positivo impar (3, 5, 7...) — mayor = desenfoque más fuerte.
blur_avg = cv2.blur(img_rgb, (7, 7))
# sigmaX=0: OpenCV infiere la dispersión del tamaño del kernel automáticamente.
blur_gauss = cv2.GaussianBlur(img_rgb, (7, 7), 0)
# Toma un único entero impar, no una tupla — a diferencia de las otras funciones de desenfoque.
blur_median = cv2.medianBlur(img_rgb, 7)
# d=9 (diámetro del vecindario), sigmaColor=75 (tolerancia de color), sigmaSpace=75 (dispersión espacial).
blur_bilateral = cv2.bilateralFilter(img_rgb, 9, 75, 75)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(img_rgb);       axes[0, 0].set_title('Original')
axes[0, 1].imshow(blur_avg);      axes[0, 1].set_title('Desenfoque Promedio')
axes[0, 2].imshow(blur_gauss);    axes[0, 2].set_title('Desenfoque Gaussiano')
axes[1, 0].imshow(blur_median);   axes[1, 0].set_title('Desenfoque Mediana')
axes[1, 1].imshow(blur_bilateral);axes[1, 1].set_title('Bilateral (conserva bordes)')
axes[1, 2].axis('off')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Kernels Personalizados

In [ ]:
# Definir kernels personalizados
kernel_sharpen = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]])
kernel_emboss = np.array([[-2, -1,  0],
                          [-1,  1,  1],
                          [ 0,  1,  2]])
kernel_identity = np.array([[0, 0, 0],
                            [0, 1, 0],
                            [0, 0, 0]])

sharpened = cv2.filter2D(img_rgb, -1, kernel_sharpen)
embossed = cv2.filter2D(gray, -1, kernel_emboss)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_rgb)
axes[0].set_title('Original')
axes[1].imshow(sharpened)
axes[1].set_title('Enfocada')
axes[2].imshow(embossed, cmap='gray')
axes[2].set_title('Relieve')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Detección de Bordes

Un **borde** es un lugar donde el brillo de los píxeles cambia bruscamente — típicamente el límite entre dos regiones u objetos. Detectar bordes es uno de los pasos más fundamentales para entender la estructura de una imagen.

Tres detectores comunes, del más simple al más robusto:

| Detector | Enfoque | Cuándo usar |
|----------|----------|-------------|
| **Sobel** | Gradiente en dirección X o Y por separado | Cuando necesitas saber la *dirección* de los bordes |
| **Canny** | Suavizar → gradiente → umbrales de histéresis | Propósito general — la elección estándar |
| **Laplaciano** | Segunda derivada en todas las direcciones a la vez | Raramente usado solo; muy sensible al ruido |

In [ ]:
# CV_64F mantiene valores negativos (los bordes pueden apuntar en ambas direcciones); usar abs() antes de mostrar.
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)  # dx=1, dy=0 → detecta bordes verticales
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)  # dx=0, dy=1 → detecta bordes horizontales
sobel_combined = cv2.magnitude(sobel_x, sobel_y)        # intensidad de borde en todas las direcciones

# Umbrales de histéresis: > 150 → siempre un borde; < 50 → descartado; en el medio → se mantiene solo
# si está conectado a un borde fuerte. Regla empírica: umbral superior ≈ 2–3× inferior.
canny = cv2.Canny(gray, 50, 150)

# Derivada de segundo orden — muy sensible al ruido; desenfoca la imagen antes de usar en la práctica.
laplacian = cv2.Laplacian(gray, cv2.CV_64F)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(gray, cmap='gray');              axes[0, 0].set_title('Original')
axes[0, 1].imshow(np.abs(sobel_x), cmap='gray');   axes[0, 1].set_title('Sobel X (bordes verticales)')
axes[0, 2].imshow(np.abs(sobel_y), cmap='gray');   axes[0, 2].set_title('Sobel Y (bordes horizontales)')
axes[1, 0].imshow(sobel_combined, cmap='gray');    axes[1, 0].set_title('Sobel Combinado')
axes[1, 1].imshow(canny, cmap='gray');             axes[1, 1].set_title('Canny')
axes[1, 2].imshow(np.abs(laplacian), cmap='gray'); axes[1, 2].set_title('Laplaciano')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Umbralización

La umbralización convierte una imagen en escala de grises en una imagen **binaria** (blanco y negro). Cada píxel se compara con un valor de corte: por encima → blanco (255), por debajo → negro (0).

El desafío es elegir el corte correcto. Tres estrategias:

| Método | Cómo se encuentra el umbral | Mejor para |
|--------|-----------------------------|---------| 
| **Fijo** | Lo especificas manualmente (p. ej. 127) | Iluminación controlada y uniforme |
| **Otsu** | Se calcula automáticamente a partir del histograma de la imagen | Imágenes con una clara separación primer plano/fondo |
| **Adaptativo** | Se calcula *por región* de la imagen | Iluminación desigual, sombras, documentos escaneados |

In [ ]:
# Umbral fijo: píxel > 127 → 255, de lo contrario → 0.
_, thresh_binary     = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
_, thresh_binary_inv = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
# Otsu: pasar threshold=0 — el valor se ignora; Otsu lo calcula automáticamente.
thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
# Adaptativo: blockSize=11 (tamaño de región local, debe ser impar), C=2 (se resta de la media local).
thresh_adaptive = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(gray, cmap='gray');            axes[0, 0].set_title('Original')
axes[0, 1].imshow(thresh_binary, cmap='gray');   axes[0, 1].set_title('Binario (127)')
axes[0, 2].imshow(thresh_binary_inv, cmap='gray');axes[0, 2].set_title('Binario Invertido')
axes[1, 0].imshow(thresh_otsu, cmap='gray');     axes[1, 0].set_title('Otsu (automático)')
axes[1, 1].imshow(thresh_adaptive, cmap='gray'); axes[1, 1].set_title('Adaptativo')
axes[1, 2].axis('off')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Operaciones Morfológicas

Las operaciones morfológicas trabajan sobre **imágenes binarias** y modifican las formas de las regiones usando una pequeña matriz llamada **elemento estructurante** (SE), que define lo que cuenta como "vecino".

Piensa en el SE como un sello: lo deslizas sobre cada píxel y aplicas una regla para decidir si ese píxel sigue siendo blanco o se vuelve negro.

| Operación | Regla | Efecto práctico |
|-----------|------|-----------------| 
| **Erosión** | Mantener blanco solo si *todos* los vecinos del SE son blancos | Reduce las regiones brillantes, elimina pequeños puntos |
| **Dilatación** | Poner blanco si *algún* vecino del SE es blanco | Crece las regiones brillantes, llena pequeños agujeros |
| **Apertura** | Erosión → Dilatación | Elimina protuberancias finas y ruido (forma conservada) |
| **Cierre** | Dilatación → Erosión | Rellena pequeños huecos y cierra contornos rotos |
| **Gradiente** | Dilatación − Erosión | Extrae el contorno de las formas |

In [ ]:
binary = thresh_otsu
# Elemento estructurante cuadrado 5×5. Para forma circular o en cruz: cv2.getStructuringElement().
kernel = np.ones((5, 5), np.uint8)
eroded   = cv2.erode(binary, kernel, iterations=1)
dilated  = cv2.dilate(binary, kernel, iterations=1)
opened   = cv2.morphologyEx(binary, cv2.MORPH_OPEN,     kernel)
closed   = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,    kernel)
gradient = cv2.morphologyEx(binary, cv2.MORPH_GRADIENT, kernel)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes[0, 0].imshow(binary,   cmap='gray'); axes[0, 0].set_title('Binario Original')
axes[0, 1].imshow(eroded,   cmap='gray'); axes[0, 1].set_title('Erosionado')
axes[0, 2].imshow(dilated,  cmap='gray'); axes[0, 2].set_title('Dilatado')
axes[1, 0].imshow(opened,   cmap='gray'); axes[1, 0].set_title('Apertura')
axes[1, 1].imshow(closed,   cmap='gray'); axes[1, 1].set_title('Cierre')
axes[1, 2].imshow(gradient, cmap='gray'); axes[1, 2].set_title('Gradiente')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Detección de Contornos

La detección de contornos va más allá de marcar píxeles de borde: agrupa píxeles de límite conectados en contornos completos. En otras palabras, la **detección de bordes** responde "¿dónde hay cambios fuertes de intensidad?", mientras que la **detección de contornos** responde "¿cuáles de esos límites forman una forma?". En la práctica, los contornos se extraen generalmente de una **imagen binaria o mapa de bordes** y son útiles cuando queremos medir o describir objetos, por ejemplo con área, perímetro o recuadros delimitadores.

In [ ]:
# Encontrar contornos
contours, hierarchy = cv2.findContours(canny, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
# Dibujar contornos sobre la imagen original
img_contours = img_rgb.copy()
cv2.drawContours(img_contours, contours, -1, (0, 255, 0), 2)
print(f"Se encontraron {len(contours)} contornos")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(canny, cmap='gray')
axes[0].set_title('Bordes Canny')
axes[1].imshow(img_contours)
axes[1].set_title(f'Contornos ({len(contours)} encontrados)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Ecualización del Histograma

Un **histograma** muestra cómo se distribuye el brillo de los píxeles en una imagen. Cuando la mayoría de los píxeles están agrupados en un rango estrecho, la imagen parece plana porque se está usando muy poco de la escala disponible de negro a blanco.

La ecualización del histograma es una herramienta práctica de recuperación de contraste:
- úsala cuando una imagen en escala de grises parece lavada o brumosa
- úsala cuando los detalles son difíciles de distinguir porque todo está en tonos grises medios
- evítala cuando el brillo ya está bien equilibrado, porque puede hacer que la imagen parezca dura

Aquí se muestran dos opciones comunes:

| Método | Mejor para | Principal limitación |
|--------|----------|-----------------| 
| **Ecualización global** | imágenes cuyo bajo contraste es similar en todas partes | puede sobreiluminar algunas regiones y aplanar otras |
| **CLAHE** | imágenes con iluminación desigual o áreas oscuras/brillantes mixtas | si se lleva demasiado lejos, puede amplificar el ruido |

En la práctica, **CLAHE** suele ser el mejor valor predeterminado porque mejora el detalle local en lugar de aplicar una corrección global a toda la imagen. El parámetro `clipLimit` evita que la mejora sea demasiado agresiva, especialmente en regiones suaves donde el ruido sobresaldría.

In [ ]:
gray = cv2.imread('../resources/images/lenna.png', cv2.IMREAD_GRAYSCALE)
# Comprimir el rango de brillo para simular una imagen de bajo contraste (brumosa).
foggy_image = (gray * 0.2 + 30).astype(np.uint8)

global_eq = cv2.equalizeHist(foggy_image)
# clipLimit=2.0: amplificación máxima por tile; tileGridSize=(8,8): imagen dividida en 64 tiles.
clahe_tool = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_res = clahe_tool.apply(foggy_image)

images = [foggy_image, global_eq, clahe_res]
titles = ['Original (bajo contraste)', 'Ecualización Global', 'CLAHE']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, (img, title) in enumerate(zip(images, titles)):
    # Fila superior: imágenes
    axes[0, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(title)
    axes[0, col].axis('off')
    # Fila inferior: histogramas — mostrar cómo el brillo se distribuye entre 0–255.
    axes[1, col].hist(img.ravel(), bins=256, range=(0, 256), color='steelblue', edgecolor='none')
    axes[1, col].set_xlim(0, 255)
    axes[1, col].set_xlabel('Brillo')
    axes[1, col].set_ylabel('Número de píxeles')
    axes[1, col].set_title(f'Histograma — {title}')
plt.tight_layout()
plt.show()

## Resumen

| Operación | Función | Caso de Uso |
|-----------|----------|----------| 
| Desenfoque Gaussiano | `cv2.GaussianBlur()` | Reducción de ruido |
| Desenfoque Mediana | `cv2.medianBlur()` | Ruido sal y pimienta |
| Bilateral | `cv2.bilateralFilter()` | Suavizado conservando bordes |
| Sobel | `cv2.Sobel()` | Detección de bordes (direccional) |
| Canny | `cv2.Canny()` | Detección de bordes (el mejor general) |
| Umbral | `cv2.threshold()` | Binarización |
| Otsu | `cv2.THRESH_OTSU` | Umbral automático |
| Erosión | `cv2.erode()` | Reducir regiones blancas |
| Dilatación | `cv2.dilate()` | Ampliar regiones blancas |
| Contornos | `cv2.findContours()` | Detección de formas |